# Room 354 Low / Medium / High Occupancy Model

This notebook builds a first-pass Room 354 occupancy-state classifier using the data already in the repo.

Important limitations:
- The target labels are **schedule-derived labels**, not CO2 threshold proxies.
- Labels are derived from the Room 354 enrollment schedule and registered counts; sensor activity is kept as an input feature only.
- Reported metrics should be read as **agreement with schedule-derived occupancy states**, not direct manual headcount accuracy.
- The optional vibration section extracts features from the two large waveform files, but those dates do not overlap the 30-day merged IAQ/HVAC window used for the base model.


In [ ]:
from __future__ import annotations

import math
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.pipeline import Pipeline

from scripts.occupancy_features import candidate_temporal_feature_columns, engineer_temporal_features

sns.set_theme(style="whitegrid")

ROOT = Path.cwd()
DATA_DIR = ROOT / "data"
REPORTS_DIR = ROOT / "reports"
ROOM_CAPACITY = 60
CLASS_ORDER = ["low", "medium", "high"]


def parse_time_token(token: str):
    return pd.to_datetime(token, format="%I%M%p").time()


def robust_norm(series: pd.Series, q_low: float = 0.05, q_high: float = 0.95) -> pd.Series:
    numeric = pd.to_numeric(series, errors="coerce")
    lo = numeric.quantile(q_low)
    hi = numeric.quantile(q_high)
    if pd.isna(lo) or pd.isna(hi) or hi <= lo:
        return pd.Series(0.0, index=numeric.index)
    return ((numeric - lo) / (hi - lo)).clip(0, 1)


def parse_enrollment_schedule(path: Path) -> pd.DataFrame:
    rows: list[dict[str, object]] = []
    for raw_line in path.read_text(encoding="utf-8", errors="replace").splitlines():
        line = raw_line.strip()
        if not line or "AIEB 354" not in line:
            continue

        tokens = line.split()
        enrollment_idx = next((i for i, token in enumerate(tokens) if re.fullmatch(r"\d+/\d+", token)), None)
        if enrollment_idx is None or len(tokens) < 5:
            continue

        enrolled, capacity = (int(value) for value in tokens[enrollment_idx].split("/"))
        start_token, end_token = tokens[3].split("-")

        rows.append(
            {
                "course": " ".join(tokens[:2]),
                "days": tokens[4],
                "start_time": parse_time_token(start_token),
                "end_time": parse_time_token(end_token),
                "enrolled": enrolled,
                "capacity": capacity,
            }
        )

    base = pd.DataFrame(rows)
    day_map = {"M": 0, "T": 1, "W": 2, "R": 3, "F": 4}
    exploded: list[dict[str, object]] = []
    for row in base.itertuples(index=False):
        for day in row.days:
            expanded = row._asdict().copy()
            expanded["day_of_week"] = day_map[day]
            expanded["start_minutes"] = row.start_time.hour * 60 + row.start_time.minute
            expanded["end_minutes"] = row.end_time.hour * 60 + row.end_time.minute
            exploded.append(expanded)
    return pd.DataFrame(exploded).sort_values(["day_of_week", "start_minutes", "course"])


def load_room354_base_dataset() -> pd.DataFrame:
    people = pd.read_csv(REPORTS_DIR / "room354_people_estimated_each_measurement.csv", parse_dates=["ts"])
    multisensor = pd.read_csv(REPORTS_DIR / "room354_multisensor_merged.csv", parse_dates=["time"])

    people_5m = (
        people.set_index("ts")[["flow_cfm", "co2_ppm", "people_est_smooth_5min"]]
        .resample("5min")
        .mean()
    )
    multisensor_5m = multisensor.set_index("time")[["voc", "humidity", "temp_avg", "occupancy_score"]]

    frame = people_5m.join(multisensor_5m, how="inner").sort_index()
    frame = frame.rename(columns={"humidity": "humidity_pct", "temp_avg": "temp_f"})
    frame["flow_lps"] = frame["flow_cfm"] * 0.47194745
    frame["delta_co2_ppm"] = (frame["co2_ppm"] - 420.0).clip(lower=0)
    frame["occ_physics_est"] = frame["people_est_smooth_5min"]
    frame["occ_physics_est_smooth"] = frame["people_est_smooth_5min"].rolling(3, min_periods=1).mean()
    frame["incoming_air_cfm"] = frame["flow_cfm"]
    return frame


def add_schedule_context(frame: pd.DataFrame, schedule: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    minute_of_day = result.index.hour * 60 + result.index.minute
    expected_enrollment: list[int] = []
    active_course_count: list[int] = []
    active_courses: list[str] = []
    is_class_time: list[bool] = []

    for ts, minute in zip(result.index, minute_of_day):
        hits = schedule[
            (schedule["day_of_week"] == ts.dayofweek)
            & (schedule["start_minutes"] <= minute)
            & (schedule["end_minutes"] > minute)
        ]
        expected_enrollment.append(int(hits["enrolled"].sum()))
        active_course_count.append(int(len(hits)))
        active_courses.append(", ".join(hits["course"].tolist()))
        is_class_time.append(not hits.empty)

    result["expected_enrollment"] = expected_enrollment
    result["expected_ratio"] = (result["expected_enrollment"] / ROOM_CAPACITY).clip(0, 1)
    result["active_course_count"] = active_course_count
    result["active_courses"] = active_courses
    result["is_class_time"] = pd.Series(is_class_time, index=result.index, dtype=float)
    return result


def assign_schedule_labels(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()

    result["sensor_intensity"] = (
        0.35 * robust_norm(result["co2_ppm"])
        + 0.25 * robust_norm(result["voc"])
        + 0.25 * robust_norm(result["people_est_smooth_5min"])
        + 0.15 * robust_norm(result["occupancy_score"])
    )

    schedule_labels: list[str] = []
    label_sources: list[str] = []
    for row in result.itertuples():
        if not bool(row.is_class_time):
            label = "low"
            source = "schedule_no_class"
        elif row.expected_ratio >= 0.70:
            label = "high"
            source = "schedule_enrollment"
        elif row.expected_ratio >= 0.30:
            label = "medium"
            source = "schedule_enrollment"
        else:
            label = "low"
            source = "schedule_enrollment"
        schedule_labels.append(label)
        label_sources.append(source)

    result["scheduled_count"] = result["expected_enrollment"].clip(0, ROOM_CAPACITY)
    result["target_schedule"] = schedule_labels
    result["target_weak"] = result["target_schedule"]
    result["label_source"] = label_sources
    return result


def build_feature_frame(base_frame: pd.DataFrame, schedule: pd.DataFrame) -> pd.DataFrame:
    temporal = engineer_temporal_features(
        base_frame[
            [
                "co2_ppm",
                "voc",
                "temp_f",
                "humidity_pct",
                "flow_cfm",
                "flow_lps",
                "delta_co2_ppm",
                "occ_physics_est",
                "occ_physics_est_smooth",
                "incoming_air_cfm",
            ]
        ],
        schedule_df=schedule[["day_of_week", "start_time", "end_time"]],
    )

    extra = base_frame[
        [
            "expected_enrollment",
            "expected_ratio",
            "active_course_count",
            "sensor_intensity",
            "scheduled_count",
            "target_schedule",
            "target_weak",
            "label_source",
        ]
    ]
    return temporal.join(extra, how="inner")


def fit_time_split_model(model_frame: pd.DataFrame, feature_columns: list[str], target_column: str = "target_schedule"):
    clean = model_frame[model_frame[target_column].notna()].copy()
    split_idx = int(len(clean) * 0.8)
    train_df = clean.iloc[:split_idx].copy()
    test_df = clean.iloc[split_idx:].copy()

    model = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=300,
                    max_depth=10,
                    min_samples_leaf=3,
                    class_weight="balanced_subsample",
                    random_state=42,
                ),
            ),
        ]
    )

    model.fit(train_df[feature_columns], train_df[target_column])
    pred = pd.Series(model.predict(test_df[feature_columns]), index=test_df.index, name="pred")

    metrics = {
        "rows_train": len(train_df),
        "rows_test": len(test_df),
        "accuracy": accuracy_score(test_df[target_column], pred),
        "macro_f1": f1_score(test_df[target_column], pred, average="macro"),
    }

    return {
        "train_df": train_df,
        "test_df": test_df,
        "feature_columns": feature_columns,
        "pipeline": model,
        "pred": pred,
        "metrics": metrics,
        "report": classification_report(test_df[target_column], pred, labels=CLASS_ORDER, zero_division=0),
    }


In [ ]:
schedule_df = parse_enrollment_schedule(DATA_DIR / "AIEB 354 Enrollment Numbers.txt")
base_df = load_room354_base_dataset()
base_df = add_schedule_context(base_df, schedule_df)
base_df = assign_schedule_labels(base_df)
model_df = build_feature_frame(base_df, schedule_df)

schedule_df.head(10)


In [ ]:
summary = pd.DataFrame(
    {
        "metric": [
            "base_rows",
            "start",
            "end",
            "schedule_label_coverage_pct",
            "unique_schedule_courses",
            "median_co2_ppm",
            "median_voc",
            "median_flow_cfm",
        ],
        "value": [
            len(base_df),
            str(base_df.index.min()),
            str(base_df.index.max()),
            round(base_df["target_schedule"].notna().mean() * 100, 2),
            schedule_df["course"].nunique(),
            round(base_df["co2_ppm"].median(), 2),
            round(base_df["voc"].median(), 2),
            round(base_df["flow_cfm"].median(), 2),
        ],
    }
)

display(summary)
display(base_df["target_schedule"].value_counts(dropna=False).rename_axis("schedule_label").to_frame("rows"))
display(base_df["label_source"].value_counts(dropna=False).rename_axis("label_source").to_frame("rows"))

fig, axes = plt.subplots(2, 1, figsize=(15, 8), sharex=True)
base_df[["co2_ppm", "voc"]].plot(ax=axes[0], title="Room 354 Sensor Levels")
base_df[["sensor_intensity", "expected_ratio"]].plot(ax=axes[1], title="Schedule-Label Drivers")
axes[1].set_ylabel("scaled value")
plt.tight_layout()


In [ ]:
all_numeric_features = candidate_temporal_feature_columns(model_df)
schedule_feature_columns = {"is_class_time", "expected_enrollment", "expected_ratio", "active_course_count", "scheduled_count"}
# The primary sensor-only model excludes schedule columns because schedule now defines the label.
sensor_only_features = [column for column in all_numeric_features if column not in schedule_feature_columns]
# This second model is a schedule-prior sanity check, not an independent validation target.
hybrid_features = sensor_only_features + ["expected_enrollment", "expected_ratio", "active_course_count"]

sensor_result = fit_time_split_model(model_df, sensor_only_features)
hybrid_result = fit_time_split_model(model_df, hybrid_features)

results_table = pd.DataFrame(
    [
        {"model": "sensor_only", **sensor_result["metrics"]},
        {"model": "sensor_plus_schedule_prior", **hybrid_result["metrics"]},
    ]
).sort_values(["macro_f1", "accuracy"], ascending=False)

display(results_table)
print("Sensor-only agreement with schedule-derived labels:\n")
print(sensor_result["report"])
print("Sensor + schedule-prior agreement with schedule-derived labels:\n")
print(hybrid_result["report"])


In [ ]:
best_result = sensor_result
test_truth = best_result["test_df"]["target_schedule"]
test_pred = best_result["pred"]

cm = confusion_matrix(test_truth, test_pred, labels=CLASS_ORDER)
cm_df = pd.DataFrame(cm, index=CLASS_ORDER, columns=CLASS_ORDER)

importances = pd.Series(
    best_result["pipeline"].named_steps["model"].feature_importances_,
    index=best_result["feature_columns"],
).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues", ax=axes[0])
axes[0].set_title("Sensor-Only Confusion Matrix")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Schedule Label")

importances.head(15).sort_values().plot(kind="barh", ax=axes[1], color="#33658A")
axes[1].set_title("Top Sensor-Only Feature Importances")
axes[1].set_xlabel("importance")

plt.tight_layout()
display(importances.head(20).rename_axis("feature").to_frame("importance"))


## Optional Vibration Feature Extraction

The two vibration files are large raw waveforms from four channels:
- `data/4-7-2026_745am-539pm.csv`
- `data/4-13-2026 754am-358pm.csv`

The helper below summarizes them in chunks into fixed windows without loading the entire file into memory. The output is suitable for later fusion with a same-day occupancy target.

Because those waveform dates do not overlap the 30-day Room 354 merged IAQ/HVAC history used above, the vibration features are prepared as an optional next step rather than forced into the base classifier.


In [ ]:
VIBRATION_CHANNELS = ["C8-1(v)", "C13-1(v)", "C8-7(v)", "C13-7(v)"]


def infer_vibration_start_datetime(path: Path) -> pd.Timestamp:
    stem = path.stem
    if "_" in stem:
        date_part, time_part = stem.split("_", 1)
    else:
        date_part, time_part = stem.split(" ", 1)

    date_ts = pd.to_datetime(date_part, format="%m-%d-%Y", errors="coerce")
    if pd.isna(date_ts):
        date_ts = pd.to_datetime(date_part, errors="raise")

    match = re.search(r"(\d{1,2})(\d{2})(am|pm)", time_part.replace(" ", ""), flags=re.IGNORECASE)
    if match is None:
        raise ValueError(f"Could not parse start time from {path.name}")

    hour = int(match.group(1))
    minute = int(match.group(2))
    suffix = match.group(3).lower()
    if suffix == "pm" and hour != 12:
        hour += 12
    if suffix == "am" and hour == 12:
        hour = 0

    return pd.Timestamp(year=date_ts.year, month=date_ts.month, day=date_ts.day, hour=hour, minute=minute)


def summarize_vibration_file(path: Path, window_seconds: int = 60, chunksize: int = 200_000) -> pd.DataFrame:
    start_ts = infer_vibration_start_datetime(path)
    accumulators: dict[int, dict[str, float]] = {}

    for chunk in pd.read_csv(path, usecols=["Time", *VIBRATION_CHANNELS], chunksize=chunksize):
        chunk = chunk.dropna(subset=["Time"]).copy()
        chunk["window_id"] = np.floor(chunk["Time"] / window_seconds).astype(int)

        for window_id, window_df in chunk.groupby("window_id"):
            state = accumulators.setdefault(int(window_id), {"count": 0.0})
            state["count"] += float(len(window_df))

            for column in VIBRATION_CHANNELS:
                values = pd.to_numeric(window_df[column], errors="coerce").dropna().to_numpy(dtype=float)
                if len(values) == 0:
                    continue
                state[f"{column}_sum_abs"] = state.get(f"{column}_sum_abs", 0.0) + float(np.abs(values).sum())
                state[f"{column}_sum_sq"] = state.get(f"{column}_sum_sq", 0.0) + float(np.square(values).sum())
                state[f"{column}_min"] = min(state.get(f"{column}_min", math.inf), float(values.min()))
                state[f"{column}_max"] = max(state.get(f"{column}_max", -math.inf), float(values.max()))

    rows: list[dict[str, float | pd.Timestamp]] = []
    for window_id, state in sorted(accumulators.items()):
        count = max(state.get("count", 0.0), 1.0)
        row: dict[str, float | pd.Timestamp] = {
            "time": start_ts + pd.to_timedelta(window_id * window_seconds, unit="s"),
            "samples": count,
        }
        total_rms = []
        total_abs = []
        total_ptp = []
        for column in VIBRATION_CHANNELS:
            sum_abs = state.get(f"{column}_sum_abs", np.nan)
            sum_sq = state.get(f"{column}_sum_sq", np.nan)
            vmin = state.get(f"{column}_min", np.nan)
            vmax = state.get(f"{column}_max", np.nan)
            mean_abs = sum_abs / count if pd.notna(sum_abs) else np.nan
            rms = math.sqrt(sum_sq / count) if pd.notna(sum_sq) else np.nan
            peak_to_peak = vmax - vmin if pd.notna(vmin) and pd.notna(vmax) else np.nan
            row[f"{column}_mean_abs"] = mean_abs
            row[f"{column}_rms"] = rms
            row[f"{column}_peak_to_peak"] = peak_to_peak
            total_rms.append(rms)
            total_abs.append(mean_abs)
            total_ptp.append(peak_to_peak)

        row["vibration_rms_mean"] = float(np.nanmean(total_rms))
        row["vibration_abs_mean"] = float(np.nanmean(total_abs))
        row["vibration_peak_to_peak_mean"] = float(np.nanmean(total_ptp))
        rows.append(row)

    return pd.DataFrame(rows).sort_values("time").reset_index(drop=True)


RUN_VIBRATION_SUMMARY = False

if RUN_VIBRATION_SUMMARY:
    vibration_files = [
        DATA_DIR / "4-7-2026_745am-539pm.csv",
        DATA_DIR / "4-13-2026 754am-358pm.csv",
    ]
    vibration_summaries = {path.name: summarize_vibration_file(path, window_seconds=60) for path in vibration_files}
    for name, summary_df in vibration_summaries.items():
        print(name, summary_df.shape)
        display(summary_df.head())


## Predict From A New Daily CSV

Set `NEW_DATA_PATH` to a CSV with these columns:
- `time`
- `co2_ppm`
- `voc`
- `temp_f`
- `humidity_pct`
- `flow_cfm`

The cell below uses the trained sensor-only model so it can predict `low`, `medium`, or `high` for each timestamp in the new file.


In [ ]:
def prepare_new_sensor_data(path: Path) -> pd.DataFrame:
    required_columns = ["time", "co2_ppm", "voc", "temp_f", "humidity_pct", "flow_cfm"]
    frame = pd.read_csv(path)
    missing = [column for column in required_columns if column not in frame.columns]
    if missing:
        missing_text = ", ".join(missing)
        raise ValueError(f"New data file is missing required columns: {missing_text}")

    frame = frame.copy()
    frame["time"] = pd.to_datetime(frame["time"], errors="coerce")
    frame = frame.dropna(subset=["time"]).sort_values("time").set_index("time")

    for column in required_columns[1:]:
        frame[column] = pd.to_numeric(frame[column], errors="coerce")

    frame["flow_lps"] = frame["flow_cfm"] * 0.47194745
    frame["delta_co2_ppm"] = (frame["co2_ppm"] - 420.0).clip(lower=0)
    frame["occ_physics_est"] = 0.0
    frame["occ_physics_est_smooth"] = 0.0
    frame["incoming_air_cfm"] = frame["flow_cfm"]

    return frame


NEW_DATA_PATH = None
# Example:
# NEW_DATA_PATH = DATA_DIR / "room354_today.csv"

if NEW_DATA_PATH is not None:
    new_sensor_df = prepare_new_sensor_data(Path(NEW_DATA_PATH))
    new_feature_df = engineer_temporal_features(
        new_sensor_df[
            [
                "co2_ppm",
                "voc",
                "temp_f",
                "humidity_pct",
                "flow_cfm",
                "flow_lps",
                "delta_co2_ppm",
                "occ_physics_est",
                "occ_physics_est_smooth",
                "incoming_air_cfm",
            ]
        ],
        schedule_df=None,
    )

    new_predictions = pd.DataFrame(index=new_feature_df.index)
    new_predictions["predicted_occupancy"] = best_result["pipeline"].predict(
        new_feature_df[best_result["feature_columns"]]
    )
    new_predictions = new_predictions.reset_index().rename(columns={"index": "time"})
    display(new_predictions.head(20))

    output_path = REPORTS_DIR / "room354_new_data_predictions.csv"
    new_predictions.to_csv(output_path, index=False)
    print(f"Saved predictions to {output_path}")
else:
    print("Set NEW_DATA_PATH to a CSV file path, then rerun this cell to generate predictions.")
